# Initialisation
Import base libraries used throughout all the stages

In [1]:
import numpy as np
import pandas as pd
import re

# Preprocessing

In [2]:
DATA_PATH = "listings.csv"
df = pd.read_csv(DATA_PATH)

VALID_TIME_FRAMES = ["ltm", "l30d", "ly"]
AMENITY_INDICATORS = [
    "Wifi",
    "Kitchen",
    "Air conditioning",
    "Washer",
    "Heating",
    "Parking",
    "Self check-in"
]
REVIEW_COLUMNS = [
    "rating",
    "accuracy",
    "cleanliness",
    "checkin",
    "communication",
    "location",
    "value"
]

We define three function that will preprocess specific attributes. 

`categorise_review_scores` discretise the reviews columns, holding a continuous numberical between 0 and 5, into categories 'Low, Medium, High'. This is done through finding the quartiles of the original set of values, and categorising any values <Q1 as 'Low', within the IQR as 'Medium', and >Q3 as 'High'. Rows without associated values are set as 'Missing'.

`frequency_of_reviews` discretise the number of reviews columns, categorising them into 'Low, Medium, High'. Similarily as before, the quartiles are found and the rows are catergorised accordingly using the same process as before.

`amenity_indicators` takes the `amenities` column, and categorises it into one of the Boolean values True or False depending on whether it contains a specific amenity (eg. wifi) within the `amenities` string.

In [3]:
def categorise_review_scores(df, review_column):
    from_column = "review_scores_" + review_column
    to_column = "review_category_" + review_column

    df = df.copy()
    data = df[from_column]

    # Calculate the first and third quartile from the original column
    Q1 = np.nanpercentile(data, 25)
    Q3 = np.nanpercentile(data, 75)

    Q3_taken = min(5.0, Q3)

    # Categorise the original review column into Low, Medium, High, according to calculated quartiles, and place it into a new column
    df[to_column] = pd.cut(
        df[from_column],
        bins=[-float("inf"), Q1, Q3_taken, float("inf")],
        labels=["Low", "Medium", "High"],
        right=False   # fix: bins include lower bounds, this also affect Q1
    )

    # Fill in the missing values for the categorised column
    df[to_column] = df[to_column].cat.add_categories('Missing')
    df[to_column] = df[to_column].fillna("Missing")

    return df


In [4]:
def frequency_of_reviews(df, time_frame = ""):
    df = df.copy()

    # Categorise 'number_of_reviews'
    if time_frame == "":
        number_of_reviews = df["number_of_reviews"]
        category_name = "review_frequency"

    # Categroise other number of reviews columns
    elif time_frame in VALID_TIME_FRAMES:
        number_of_reviews = df["number_of_reviews_" + time_frame]
        category_name = "review_frequency_" + time_frame
    else:
        # This should never happen
        raise ValueError("Invalid time frame")

    # Calculate percentiles, without accounting for listings with 0 reviews
    Q1 = np.nanpercentile(number_of_reviews[number_of_reviews > 0], 25)
    Q3 = np.nanpercentile(number_of_reviews[number_of_reviews > 0], 75)

    # Create a new column to categorise
    df[category_name] = pd.cut(
        number_of_reviews,
        bins=[-float("inf"), 0, Q1, Q3, float("inf")],
        labels=["No Reviews","Low", "Medium", "High"]
    )

    return df

In [5]:
def amenity_indicators(df):
    df = df.copy()

    for amenity in AMENITY_INDICATORS:
        column_name = "has_" + amenity.lower().replace(" ", "_")

        # Create a new column determining whether each listing has our desired amenity
        df[column_name] = df["amenities"].apply(
            lambda x: bool(re.search(amenity, str(x), re.IGNORECASE))
        )

    return df

We pass on our read dataframe to each of the functions, to include columns including our processed values.

In [6]:
df = pd.read_csv(DATA_PATH)

# Discretise the review columns
for column in REVIEW_COLUMNS:
    df = categorise_review_scores(df, column)

# Categorise the 'number_of_reviews' column
df = frequency_of_reviews(df)

# Categorise all other number of reviews columns
for time_frame in VALID_TIME_FRAMES:
    df = frequency_of_reviews(df, time_frame)

# Determine if certain amenities exist
df = amenity_indicators(df)

Finally, we save the processed data into a seperate csv file.

In [7]:
df.to_csv("processed_data.csv", index=False)

# Correlation Analysis

In [8]:
import json
from itertools import combinations
from sklearn.preprocessing import KBinsDiscretizer
from scipy.stats import pearsonr, spearmanr
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression

DATA_PATH = "processed_data.csv"   # output of preprocessing_2.py, NOT processed_only.csv
REFERENCE_DATE = pd.Timestamp("2026-07-07")  # set this to your dataset's scrape date
 
AMENITY_INDICATORS = [
    "Wifi", "Kitchen", "Air conditioning", "Washer", "Heating", "Parking", "Self check-in"
]
REVIEW_FREQ_ORDER = ["No Reviews", "Low", "Medium", "High"]

df = pd.read_csv(DATA_PATH)
df['price'] = df['price'].replace(r'[\$,]', '', regex=True).astype(float)

In [9]:
# Creating an amernity count column from preproccessed data
has_cols = ["has_" + a.lower().replace(" ", "_") for a in AMENITY_INDICATORS]
df["amenity_count"] = df[has_cols].sum(axis=1)

# Reading in review frequency data and converting to an ordered categorical variable
df["review_frequency"] = pd.Categorical(
    df["review_frequency"], categories=REVIEW_FREQ_ORDER, ordered=True
).codes.astype(float)
df.loc[df["review_frequency"] < 0, "review_frequency"] = np.nan

variables = {"review_scores_rating": ("review_scores_rating", "numeric"),
    "price": ("price", "numeric"),
    "amenity_count": ("amenity_count", "numeric"),
    "host_listings_count": ("host_listings_count", "numeric"),
    "review_frequency": ("review_frequency", "ordinal")}

df = df.dropna(subset=list(variables.keys()))

In [10]:
def mutual_information(x, y, type_x, type_y):
    x = np.asarray(x, dtype=float).reshape(-1, 1)
    y = np.asarray(y, dtype=float)

    if type_y == "ordinal":
        return mutual_info_classif(x, y.astype(int), discrete_features=False, random_state=42)[0]
    if type_x == "ordinal":
        return mutual_info_classif(y.reshape(-1, 1), x.ravel().astype(int),
                                   discrete_features=False, random_state=42)[0]
    return mutual_info_regression(x, y, discrete_features=False, random_state=42)[0]

In [11]:
results = []
for v1, v2 in combinations(variables, 2):
# Pearson correlation
    pearson_corr, _ = pearsonr(df[v1], df[v2])

    # Spearman correlation
    spearman_corr, _ = spearmanr(df[v1], df[v2])
        
    # Mutual information (no discretising)
    mi = mutual_information(df[v1], df[v2], variables[v1][1], variables[v2][1])
        
   
    print(f"Correlation between {v1} and {v2}")
    print(f"  Pearson: {pearson_corr:.4f}")
    print(f"  Spearman: {spearman_corr:.4f}")
    print(f"  Mutual Information: {mi:.4f}\n")


    results.append({
        "var_1": v1,
        "var_2": v2,
        "pearson": round(pearson_corr, 4),
        "spearman": round(spearman_corr, 4),
        "mutual_information": round(float(mi), 4),
    })

Correlation between review_scores_rating and price
  Pearson: 0.0292
  Spearman: 0.0921
  Mutual Information: 0.0218

Correlation between review_scores_rating and amenity_count
  Pearson: 0.0792
  Spearman: 0.0212
  Mutual Information: 0.0170

Correlation between review_scores_rating and host_listings_count
  Pearson: -0.2168
  Spearman: -0.2783
  Mutual Information: 0.1276

Correlation between review_scores_rating and review_frequency
  Pearson: 0.2052
  Spearman: -0.0937
  Mutual Information: 0.3589

Correlation between price and amenity_count
  Pearson: 0.0339
  Spearman: 0.2511
  Mutual Information: 0.0637

Correlation between price and host_listings_count
  Pearson: -0.0160
  Spearman: 0.0574
  Mutual Information: 0.3016

Correlation between price and review_frequency
  Pearson: -0.0119
  Spearman: 0.0153
  Mutual Information: 0.0366

Correlation between amenity_count and host_listings_count
  Pearson: 0.0482
  Spearman: 0.0484
  Mutual Information: 0.0732

Correlation between ame

In [12]:
with open("correlation_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved results to correlation_results.json")

Saved results to correlation_results.json


# Supervised Learning

In [13]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.dummy import DummyClassifier

DATA_PATH = 'processed_data.csv'
 
Y_COL = 'review_category_rating'
X_COLS = ['host_listings_count', 'latitude', 'longitude', 'accommodates',
          'number_of_reviews', 'availability_365', 'reviews_per_month', 'host_has_profile_pic',
          'hosts_time_as_user_years', 'host_is_superhost', 'host_identity_verified',
          'has_wifi', 'has_kitchen', 'has_air_conditioning', 'has_washer', 'has_heating',
          'has_parking', 'has_self_check-in', 'review_frequency', 'price']

BOOL = ['host_is_superhost', 'host_identity_verified', 'host_has_profile_pic',]
CAT = ['review_frequency', 'has_wifi', 'has_kitchen', 'has_air_conditioning', 'has_washer', 'has_heating',
          'has_parking', 'has_self_check-in']
STR = ['price']
 
FILTER_CATEGORIES = ['Missing']

TEST_SPLIT = 0.2

KNN_N_NEIGHBORS = 5

df = pd.read_csv(DATA_PATH)

We define functions that converts non-numberical columns into numerical columns.

Boolean columns turn into 1 or 0. Categorical values get converted into integers, depending on how many categories there are.

The `price` column is also parsed to convert it from a string to a numberical value.

In [14]:
def process_bool(val):
    if pd.isna(val):
        return np.nan

    if val[0].lower() == 't':
        return 1
    elif val[0].lower() == 'f':
        return 0



def process_cat(cat, cat_dict):
    return cat_dict[cat]



def parse_price(price):
    # Check if the current price is a string
    if not pd.isna(price):
        # Remove the `$` and `,` symbols from the string
        cleaned_price = re.sub(r'[$,]', '', price)
        return (float(cleaned_price))
    else:
        # If price is Nan, just keep it as NaN
        return price

In [15]:
# Process Booelan Values
for column in BOOL:
    df[column] = df[column].apply(process_bool)

# Process Categorical Values
for column in CAT:
    categories = list(set(df[column].values))
    cat_dict = {}

    for i in range(len(categories)):
        cat_dict[categories[i]] = i

    df[column] = df[column].apply(process_cat, cat_dict=cat_dict)


# Process price
df['price'] = df['price'].apply(parse_price)


# Filter out the rows with 'missing'
for category in FILTER_CATEGORIES:
    df = df[df[Y_COL] != category]

We split the data into train and test sets. The ratio was chosen to be 80/20.

In [16]:
# Split into train and test sets

X = df[X_COLS]
y = df[Y_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SPLIT,
    stratify=y,
    random_state=42
)

In [17]:
def impute_and_scale(X_train, X_test):
    num_cols = X_train.select_dtypes(include='number').columns

    # Impute with median values
    imputer = SimpleImputer(missing_values=np.nan, strategy='median')
    X_train[num_cols] = imputer.fit_transform(X_train[num_cols])
    X_test[num_cols] = imputer.transform(X_test[num_cols]) 

    # Scale the values for each column
    scaler = StandardScaler()
    X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test[num_cols] = scaler.transform(X_test[num_cols])
 
    return X_train, X_test

# Fill out the missings values (impute), scale the values
X_train, X_test = impute_and_scale(X_train, X_test)

In [ ]:
def run_knn(X_train, X_test, y_train, y_test):
    k_values = range(1, 200, 5)
    five_f_CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    best_k = 0
    best_score = -1

    for k in k_values:  
        results = []

        for train_idx, test_idx in five_f_CV.split(X_train, y_train):
            X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[test_idx]
            y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[test_idx]

            knn = KNeighborsClassifier(n_neighbors=k)
            knn.fit(X_tr, y_tr)

            y_pred = knn.predict(X_val)
            results.append(accuracy_score(y_val, y_pred))

        mean_score = np.mean(results)
        print(f"k={k}: mean accuracy = {mean_score:.4f}")

        if mean_score > best_score:
            best_score = mean_score
            best_k = k

    print(f"\nBest k: {best_k}, score: {best_score}")

    best_knn = KNeighborsClassifier(n_neighbors=best_k)
    best_knn.fit(X_train, y_train)

    return best_knn

knn = run_knn(X_train, X_test, y_train, y_test)

k=1: mean accuracy = 0.5816
k=6: mean accuracy = 0.6328
k=11: mean accuracy = 0.6471
k=16: mean accuracy = 0.6495
k=21: mean accuracy = 0.6532
k=26: mean accuracy = 0.6549
k=31: mean accuracy = 0.6572
k=36: mean accuracy = 0.6558


In [ ]:
def run_decision_tree(X_train, X_test, y_train, y_test):
    depth_values = range(1, 20)
    n = 5
    nf_CV = StratifiedKFold(n_splits=n, shuffle=True, random_state=42)

    best_depth = 0
    best_score = -1

    for depth in depth_values:
        results = []

        for train_idx, test_idx in nf_CV.split(X_train, y_train):
            X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[test_idx]
            y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[test_idx]

            dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
            dt.fit(X_tr, y_tr)

            y_pred = dt.predict(X_val)
            results.append(accuracy_score(y_val, y_pred))

        mean_score = np.mean(results)
        print(f"max_depth={depth}: mean accuracy = {mean_score:.4f}")

        if mean_score > best_score:
            best_score = mean_score
            best_depth = depth

    print(f"\nBest max_depth: {best_depth}, score: {best_score:.4f}")

    best_dt = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
    best_dt.fit(X_train, y_train)

    return best_dt

dt = run_decision_tree(X_train, X_test, y_train, y_test)

In [ ]:
def feature_importance(model, feature_names):
    importances = pd.Series(model.feature_importances_, index=feature_names)
    importances = importances.sort_values(ascending=False)

    print("\nFeature importances (Decision Tree):")
    print(importances)

    return importances

feature_importance(dt, X_train.columns)

In [ ]:
def baseline(X_train, X_test, y_train, y_test):
    dummy = DummyClassifier(strategy='most_frequent')
    dummy.fit(X_train, y_train)

    y_pred = dummy.predict(X_test)
    score = accuracy_score(y_test, y_pred)

    print(f"Baseline (majority class) accuracy: {score:.4f}")

    return score


def bootstrap_ci(model, X_test, y_test, n_iterations=10):
    scores = []
    n = len(X_test)

    for i in range(n_iterations):
        idx = np.random.choice(n, size=n, replace=True)
        X_sample = X_test.iloc[idx]
        y_sample = y_test.iloc[idx]

        y_pred = model.predict(X_sample)
        scores.append(accuracy_score(y_sample, y_pred))

    scores = np.array(scores)
    lower = np.percentile(scores, 2.5)
    upper = np.percentile(scores, 97.5)

    print(f"Bootstrap 95% CI: [{lower:.4f}, {upper:.4f}] (mean={scores.mean():.4f})")

    return lower, upper

In [ ]:
# Baseline
baseline_score = baseline(X_train, X_test, y_train, y_test)

knn_test_score = accuracy_score(y_test, knn.predict(X_test))
dt_test_score = accuracy_score(y_test, dt.predict(X_test))

print(f"\nKNN test accuracy: {np.round(knn_test_score),4} (baseline: {np.round(baseline_score,4)}")
print(f"Decision Tree test accuracy: {np.round(dt_test_score,4)} (baseline: {np.round(baseline_score,4)}")

bootstrap_ci(knn, X_test, y_test)

In [ ]:
path='predictions.csv'
models = {'knn': knn, 'decision_tree': dt}


output = pd.DataFrame(index=X_test.index)
output['actual'] = y_test

for name, model in models.items():
    output[f'predicted_{name}'] = model.predict(X_test)

output.to_csv(path, index=True)
print(f"\nSaved predictions to {path}")